I apologize for the state of this homework submission; it is rather messy.
Work has been busy and I started one week late, so I just tried to do the minimum, based on the notes I took during module 1 to help me get to answering the questions. Thanks heaps for running this course!

1.2 Enviroment

In [37]:
from dotenv import load_dotenv
load_dotenv()

True

In [38]:
from openai import OpenAI
openai_client = OpenAI()

1.3 RAG (Concept of..)

In [39]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [40]:
query = "How does the agentic loop keep calling the model until it stops?"

#answer = llm(question)
#print(answer)

def rag(question):

    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

1.4 Dataset (forming the knowledge base)

In [41]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [42]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [43]:
print(len(documents))

72


In [44]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [45]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [46]:
print(len(chunks))

295


In [47]:
chunks[294]

{'start': 1000,
 'content': 'c.\n\n):\n\n1. Split it into chunks\n2. Evaluate the same way as multiple articles\n3. You can use `youtube-transcript-api` to get transcripts\n   programmatically\n\n## Book or very long content\n\nFor books and other long-form content, apply this strategy:\n\n1. Treat each chapter or section as a separate document\n2. Experiment with different chunking strategies\n3. Use LLM-as-a-Judge to compare approaches\n\n## Images and slides\n\nVisual content can be processed as follows:\n\n1. Describe images using an LLM like GPT-4o-mini\n2. Each image is a separate document\n3. For slide decks: deck = document, slide = chunk\n4. You can also use CLIP embeddings for direct image search\n\n## Smart chunking with LLMs\n\nInstead of splitting by character count or paragraph breaks, you\ncan use an LLM to find logical boundaries:\n\n1. Give the LLM the full text and ask it to split into logical\n   blocks\n2. Then ask it to name each block\n3. Each block becomes a chun

1.5 Search (+Indexing)

CONCEPTS:

-Searching narrows down the context you are feeding to the llm - less token more accuraracy.

-To search better, you need to index your knowledge base ("documents" variable) first

-Indexing: "organize" your knowledge base into keyword fields and text fields.
 Keyword fields for absolute filtering. 
 Text fields for text search.

-Search: 2 kinds, 
 text/lexical: # of words that matches
 vector/semantic: "meaning", done by llm.



STEPS:

-Create "index" variable

-Index our knowledgebase

-Create "search" function with "boosting field" and "filter"

In [48]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [49]:
def search(query):
    
    return index.search(
        query,
        num_results=5
    )

In [50]:
search_results = search(query)

In [51]:
search_results[0]

{'start': 4000,
 'content': 'while` loop. The loop keeps calling the model until\nit returns a response without any function calls. We also keep an\niteration counter so we can see how many round-trips happened.\n\n```python\nit = 1\n\nwhile True:\n    print(f"iteration #{it}...")\n    has_function_calls = False\n\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=messages,\n        tools=[search_tool],\n    )\n\n    messages.extend(response.output)\n\n    for item in response.output:\n        if item.type == "function_call":\n            print("function_call:", item.name, item.arguments)\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n\n        elif item.type == "message":\n            print("ASSISTANT:")\n            print(item.content[0].text)\n\n    it = it + 1\n    if has_function_calls == False:\n        break\n```\n\nThis is the core agent loop. The model reaso

1.6: Building Prompt

CONCEPT:

-Prompt for llms consists of [Instruction][User Prompt]

-[User Prompt]=The bit that changes everytime(query). 
In our example, this would be the "question" that users ask, 

and "context"-which is the narrowed-down information we retrieve from our knowledge base, to reduce the workload of the llm.


STEPS:

-Define datatype USER_PRMPT_TEMPLATE.

-Build function "build_context" to call "search" built in module 1.5, then convert the retrieved data into a FORM EASY FOR LLM o read.

-Build Function "build_prompt", calls "built_context", put this and the "question" input inside the USER_PROMPT_TEMPLATE datatype.


In [52]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [53]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [54]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["content"])
        lines.append("")

    return "\n".join(lines).strip()

In [55]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [56]:
prompt = build_prompt(query, search_results)

print(prompt)

Question:
How does the agentic loop keep calling the model until it stops?

Context:
while` loop. The loop keeps calling the model until
it returns a response without any function calls. We also keep an
iteration counter so we can see how many round-trips happened.

```python
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break
```

This is the core agent 

1.7 LLM

CONCEPT:

-different LLM/API produces different outpts.
-we use OPENAI's "response" API. The response (output) is Pydantic - matches a preset RULE / DATA TYPE. 

-It can contain interesting/useful info like how much token you used!!

-A PROPER PROMPT is consisted of INSTRUCTIONS and PROMPT. 
 We need to include the message history in the prompt - otherwise LLM wont work.

 e.g: if we frist say "we live in Auckland", then ask "what's the weather like?" - if we dont include the history, llm wont know we are asking about weather in Auckland.

 -Seperating INSTRUCTIONS from PROMPTS also prevent PROMPT INJECTION ATTACKS, where hackers literally give instruction in the prompt to ignore security rules.

STEPS:

-Create datatype "message_history"

-Create the function "llm", which creats the data package in the format OPEN AI likes, 
 INSTRUCTION, and PROMPT based on "message_history"

-Finally, we create "rag", which is our entire RAG flow:
 -SEARCH: narrowed down context from indexed knowledge base
 -PROMPT: user question + SEARCH formated into LLM-prefered style + message_history
 -LLM: wraps INSTRUCTION + PROMPT into OPEN-AI data capsule, makes query to OPEN-AI and tell it which LLM model we want to use. 


In [57]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [58]:
response.output[0]

ResponseOutputMessage(id='msg_000fb239a34a3109006a38ff82a0c081998bff33e25b209efa', content=[ResponseOutputText(annotations=[], text='It keeps calling the model with a `while True` loop and only stops when the model returns **no `function_call` items** in that turn.\n\nIn your code, the logic is:\n\n1. Call the model.\n2. Inspect `response.output`.\n3. If any item is a `function_call`:\n   - execute the tool\n   - append the tool result to `messages`\n   - set `has_function_calls = True`\n4. If there were **no** function calls, break out of the loop.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nThat means:\n- **function call present** → keep looping\n- **no function calls** → model has given its final answer, stop\n\nIn short: **the agentic loop continues until the model stops asking for tools.**', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [59]:
response.usage

ResponseUsage(input_tokens=2168, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=179, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2347)

In [60]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [61]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [62]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [63]:
answer = rag(query)
print(answer)

It keeps looping with a `while True` loop. Each turn:

1. Calls the model.
2. Checks the output for any `function_call` items.
3. If there are function calls, it runs them and adds the results back to `messages`.
4. If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the model’s response**.


In [64]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the knowledge database for entries matching the given question.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Search question text to look up in the knowledge database."
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [65]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [66]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the knowledge database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [68]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [69]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the knowledge database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [70]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [71]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [72]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)
print(result)

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content='\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n', role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop plain RAG difference context"}', call_id='call_d2QJZI6J9Awgt6iCvVAnB0Q7', name='search', type='function_call', id='fc_08e6d524c66715fa006a38ff876f0c8198aac604d338e84203', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_d2QJZI6J9Awgt6iCvVAnB0Q7', 'output': '[\n  {\n    "start": 4000,\n    "content": "esult.\\n\\nThe `result` is a `LoopResult` with `all_messages` (the full\\nconversation), token counts, and `cost` (co

In [79]:
search_call_count = sum(
    1 for msg in result.all_messages  # <-- Added .messages here
    if isinstance(msg, dict) and msg.get("role") == "tool" and msg.get("name") == "search"
)

print(f"The 'search' tool was called {search_call_count} times.")

The 'search' tool was called 0 times.


In [78]:
print(dir(result))

['__annotations__', '__class__', '__class_getitem__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'all_messages', 'cost', 'last_message', 'new_messages', 'tokens']
